### Controlled predictor analysis — Condition & Vegetation response

For each of the six predictors examined in the grouped-bar figures
(berm length, slope, landform, soil texture, B-horizon presence,
flow accumulation), this notebook asks:

> **Does the predictor remain significant after controlling for
> the other five?**

Methods per predictor:

1. **Univariate association (chi-square / Fisher exact / univariate logistic)**
   These tests assess whether a single predictor is related to the outcome *without adjusting for anything else*. Chi-square and Fisher exact tests treat both the predictor and outcome as categorical — Fisher exact is the small-sample alternative when expected cell counts drop below about 5. Univariate logistic regression serves the same purpose but can handle continuous predictors (like flow accumulation or slope) without binning them, and it yields an odds ratio that quantifies the direction and magnitude of the effect. This step establishes a baseline: does the predictor matter at all in isolation?

2. **Multivariate logistic regression — focal predictor coefficient**
   All six predictors are entered simultaneously into a single logistic regression model. The coefficient (and corresponding odds ratio) for each predictor now reflects its association with the outcome *after holding the other five constant*. A predictor whose coefficient is significant here has explanatory power that is not redundant with the other variables. Comparing univariate and multivariate coefficients reveals confounding: if a predictor is significant univariately but not in the full model, its apparent effect was being driven by correlated covariates.

3. **Likelihood-ratio test (full model vs model without focal predictor)**
   For each focal predictor we fit a *reduced* model that drops only that predictor's term(s) and compare it to the full model using a likelihood-ratio χ² test. The test statistic is $LR = -2(\ell_{\text{reduced}} - \ell_{\text{full}})$, which follows a χ² distribution with degrees of freedom equal to the number of coefficients removed (1 for a continuous or binary predictor; more for multi-level categoricals). A significant LRT means the full model fits meaningfully better, so the focal predictor contributes unique information beyond the other five. We also report ΔAIC = AIC(reduced) − AIC(full); values above +2 indicate a meaningful independent contribution.

4. **Random-forest permutation importance**
   As a model-agnostic robustness check, we fit a 500-tree random forest (max depth 5, balanced class weights) with all predictors and then measure each predictor's importance by randomly permuting its values and recording the drop in accuracy (averaged over 30 repeats). Predictors whose permutation causes a large accuracy drop are important to the model regardless of linearity assumptions or collinearity structure. Five-fold stratified cross-validated AUC provides an overall measure of the model's discriminative ability.

Both outcomes are analysed:
- **Berm condition** (Intact vs Degraded)
- **Vegetation response** (Response vs No response)


In [ ]:
import sys as _sys
_sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import fisher_exact, chi2_contingency, chi2 as chi2_dist
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

from constants import LBL_EFFECTIVE, LBL_INEFFECTIVE
from analysis import PRETTY_LABELS


In [ ]:
data = pd.read_csv('../data/merged.csv')

# Binary outcome columns
data['_intact_01'] = data['Intact'].astype(int)
data['_effective_01'] = (data['Effective'] == LBL_EFFECTIVE).astype(int)

# Binary encoding for categorical predictors used in the model
data['_has_B'] = (data['Soil_Development'] == 'B horizon').astype(int)

print(f"Loaded {len(data):,} berms")
print(f"  Intact rate:     {data['_intact_01'].mean():.1%}")
print(f"  Effective rate:  {data['_effective_01'].mean():.1%}")


## Methodology

### Motivation

The grouped-bar figures (Figs 4–5) test each predictor's univariate association with berm condition and vegetation response — Fisher exact tests for two-category comparisons (e.g., short vs long berms) and chi-square tests for multi-category predictors (e.g., landform, soil texture). However, univariate tests cannot distinguish direct effects from confounded effects. For example, landform covaries with slope, soil texture, and flow accumulation — so a significant univariate landform effect may really reflect slope differences between landforms.

To isolate each predictor's *independent* contribution we use a *drop-one likelihood-ratio test (LRT)* framework:

1. Fit a *full logistic regression* with all seven predictors entering simultaneously.
2. For each focal predictor, fit a *reduced model* that drops only that predictor.
3. Compare the two models with a likelihood-ratio χ² test (df = number of coefficients removed).

Each model has a log-likelihood — a measure of how probable the observed data are under that model. The LRT statistic is $LR = -2(\ell_{\text{reduced}} - \ell_{\text{full}})$, which is always ≥ 0 because adding parameters can only improve or maintain fit. Under the null hypothesis that the removed predictor has no effect, $LR$ follows a χ² distribution with degrees of freedom equal to the number of coefficients dropped (1 for a continuous or binary predictor, 2 for landform, 7 for soil texture). A large $LR$ (small p-value) means the full model fits meaningfully better, so the focal predictor contributes information beyond what the other predictors already explain.

This directly answers: *does including predictor X improve the model beyond what the other six predictors already explain?*

### Full model specification

$$\text{logit}\bigl(P(Y=1)\bigr) = \beta_0 + \beta_1\,\text{Shape\_Leng} + \beta_2\,\text{slope\_200} + \beta_3\,\text{FA\_30\_max} + \beta_4\,\text{claytotal\_r} + \beta_5\,\text{has\_B} + \boldsymbol{\gamma}\,\text{Landform} + \boldsymbol{\delta}\,\text{Texture}$$

| Predictor | Type | Encoding | df |
|---|---|---|---|
| Berm length (`Shape_Leng`) | Continuous | As-is (metres) | 1 |
| Slope (`slope_200`) | Continuous | As-is (%, 200 m buffer) | 1 |
| Flow accumulation (`FA_30_max`) | Continuous | As-is (30 m contributing cells) | 1 |
| Clay content (`claytotal_r`) | Continuous | As-is (%) | 1 |
| B-horizon presence (`Soil_Development`) | Binary | 1 = B horizon, 0 = No B horizon | 1 |
| Landform | Categorical (3 levels) | One-hot, drop-first | 2 |
| Soil texture | Categorical (8 levels) | One-hot, drop-first | 7 |

The model is fit twice — once with $Y$ = Intact (berm condition) and once with $Y$ = Vegetation response.

### Measures reported

- **Univariate p**: chi-square test of independence (categorical predictors) or univariate logistic regression (continuous predictors). Indicates raw association *before* controlling. 
Note: the grouped-bar figures use Fisher exact tests for pairwise comparisons, whereas this notebook uses chi-square for categoricals because all sample sizes are large.
- **LRT p**: likelihood-ratio test comparing full vs reduced model. Indicates *independent* contribution after controlling for the other six predictors.
- **ΔAIC**: AIC(reduced) − AIC(full). Positive = the focal predictor improves model fit; values > 2 are considered meaningful.
- **Controlled OR (95 % CI)**: the odds ratio is $OR = e^{\hat\beta}$, where $\hat\beta$ is the focal predictor's coefficient in the full model. 
  - This gives the multiplicative change in the odds of the outcome for a one-unit increase in the predictor, holding all other predictors constant. An OR > 1 means higher values of the predictor increase the odds; OR < 1 means they decrease the odds; OR = 1 means no effect. The 95 % confidence interval is $\bigl(e^{\hat\beta - 1.96\,SE},\; e^{\hat\beta + 1.96\,SE}\bigr)$; if the confidence interval excludes 1 the effect is significant at α = 0.05. Only single-coefficient predictors (continuous or binary) yield one OR. Multi-level categoricals (landform, texture) have multiple dummy coefficients, so we report their **joint LRT** instead — a single test of whether *all* of the variable's dummy coefficients are simultaneously zero ($H_0\!: \delta_1 = \delta_2 = \cdots = \delta_k = 0$). A significant joint LRT means at least one category differs from the reference level after controlling, but does not identify which one.
- **RF permutation importance**: model-agnostic check via a 500-tree random forest (`max_depth=5`, balanced classes, 30 permutation repeats). CV AUC gives overall discriminative ability.

In [ ]:
# ── Predictor definitions ─────────────────────────────────────────────────
_ALL_PRED_COLS = ['Shape_Leng', 'slope_200', 'FA_30_max', 'claytotal_r',
                  'Landform', 'Texture', 'Soil_Development']

_FOCAL_PREDICTORS = [
    ('Shape_Leng',       'Berm length'),
    ('slope_200',        'Slope'),
    ('Landform',         'Landform'),
    ('Texture',          'Soil texture'),
    ('Soil_Development', 'B-horizon presence'),
    ('FA_30_max',        'Flow accumulation'),
    ('claytotal_r',      'Clay content'),
]

_CATEGORICALS = {'Landform', 'Texture', 'Soil_Development'}


def _prepare_model_matrix(df, y_col):
    """Complete-cases design matrix with dummies."""
    cols = [y_col] + _ALL_PRED_COLS
    mdf = df[cols].dropna().copy()
    mdf['_has_B'] = (mdf['Soil_Development'] == 'B horizon').astype(int)
    mdf = mdf.drop(columns=['Soil_Development'])
    cat_cols = [c for c in ['Landform', 'Texture'] if c in mdf.columns]
    X = pd.get_dummies(mdf.drop(columns=[y_col]),
                       columns=cat_cols, drop_first=True).astype(float)
    X = sm.add_constant(X)
    y = mdf[y_col].astype(float)
    return X, y, mdf


def _focal_model_cols(X, focal):
    """Column name(s) in X that correspond to a focal predictor."""
    if focal == 'Soil_Development':
        return ['_has_B']
    if focal in _CATEGORICALS:
        return [c for c in X.columns if c.startswith(focal + '_')]
    return [focal]


def _univariate(df, focal, y_col):
    """Univariate association test."""
    d = df[[focal, y_col]].dropna()
    if focal in _CATEGORICALS:
        ct = pd.crosstab(d[focal], d[y_col])
        chi2_val, p, dof, _ = chi2_contingency(ct)
        n = ct.sum().sum()
        v = np.sqrt(chi2_val / (n * (min(ct.shape) - 1)))
        return {'test': 'chi-square', 'stat': chi2_val, 'p': p,
                'effect': f"V = {v:.3f}", 'n': n}
    X_u = sm.add_constant(d[focal].astype(float))
    fit = sm.Logit(d[y_col].astype(float), X_u).fit(disp=0)
    coef = fit.params[focal]
    return {'test': 'logistic', 'p': fit.pvalues[focal],
            'effect': f"OR = {np.exp(coef):.3f}", 'n': len(d)}


def controlled_predictor_analysis(df, y_col, outcome_label):
    """
    Full controlled analysis for each focal predictor.
    Returns (summary_df, coef_df, importance_df).
    """
    X, y, mdf = _prepare_model_matrix(df, y_col)
    n = len(y)

    # ── Full model ────────────────────────────────────────────────────────
    logit_full = sm.Logit(y, X).fit(disp=0)
    print(f"\n{'═'*70}")
    print(f"  {outcome_label}")
    print(f"  Full model: n = {n},  Pseudo R² = {logit_full.prsquared:.4f},"
          f"  AIC = {logit_full.aic:.1f}")
    print(f"{'═'*70}")

    # ── Full coefficient table ────────────────────────────────────────────
    coef_df = pd.DataFrame({
        'coef':  logit_full.params,
        'OR':    np.exp(logit_full.params),
        'SE':    logit_full.bse,
        'z':     logit_full.tvalues,
        'p':     logit_full.pvalues,
    }).drop(index='const')
    coef_df['sig'] = coef_df['p'].apply(
        lambda p: '***' if p < 0.001 else '**' if p < 0.01
                  else '*' if p < 0.05 else 'ns')
    coef_df = coef_df.sort_values('z', key=abs, ascending=False)

    # ── Per-focal LRT ─────────────────────────────────────────────────────
    rows = []
    for focal, label in _FOCAL_PREDICTORS:
        uni = _univariate(df, focal, y_col)
        focal_cols = _focal_model_cols(X, focal)
        X_red = X.drop(columns=focal_cols)
        logit_red = sm.Logit(y, X_red).fit(disp=0)
        lr_stat = -2 * (logit_red.llf - logit_full.llf)
        lr_df   = len(focal_cols)
        lr_p    = chi2_dist.sf(lr_stat, df=lr_df)
        d_aic   = logit_red.aic - logit_full.aic

        # Extract OR for single-column focal predictors
        if len(focal_cols) == 1:
            fc = focal_cols[0]
            c  = logit_full.params[fc]
            se = logit_full.bse[fc]
            or_str = (f"{np.exp(c):.3f} "
                      f"({np.exp(c - 1.96*se):.3f}\u2013{np.exp(c + 1.96*se):.3f})")
            p_coef = f"{logit_full.pvalues[fc]:.4f}"
        else:
            or_str = f"({lr_df} df joint)"
            p_coef = "\u2014"

        sig_u = ('***' if uni['p'] < 0.001 else '**' if uni['p'] < 0.01
                 else '*' if uni['p'] < 0.05 else 'ns')
        sig_l = ('***' if lr_p < 0.001 else '**' if lr_p < 0.01
                 else '*' if lr_p < 0.05 else 'ns')
        print(f"  {label:25s}  Uni p={uni['p']:.4f} ({sig_u:3s})  "
              f"LRT={lr_stat:6.2f} (df={lr_df}, p={lr_p:.4f}, {sig_l:3s})  "
              f"\u0394AIC={d_aic:+.1f}")

        rows.append({
            'Predictor': label,
            'Column': focal,
            'n': uni['n'],
            'Univariate effect': uni['effect'],
            'Univariate p': uni['p'],
            'LRT': lr_stat,
            'LRT df': lr_df,
            'LRT p': lr_p,
            'Controlled OR (95% CI)': or_str,
            'Coef p': p_coef,
            '\u0394AIC': d_aic,
        })

    summary = pd.DataFrame(rows)

    # ── Random forest ─────────────────────────────────────────────────────
    X_rf = X.drop(columns=['const'])
    rf = RandomForestClassifier(n_estimators=500, max_depth=5,
                                random_state=42, class_weight='balanced')
    rf.fit(X_rf, y)
    cv = cross_val_score(rf, X_rf, y,
                         cv=StratifiedKFold(5, shuffle=True, random_state=42),
                         scoring='roc_auc')
    perm = permutation_importance(rf, X_rf, y, n_repeats=30,
                                  random_state=42, scoring='accuracy')
    imp = pd.DataFrame({
        'importance': perm.importances_mean,
        'std':        perm.importances_std,
    }, index=X_rf.columns).sort_values('importance', ascending=False)
    print(f"\n  RF 5-fold CV AUC = {cv.mean():.3f} \u00b1 {cv.std():.3f}")

    return summary, coef_df, imp


## Berm condition (Intact vs Degraded)


In [ ]:
_cond_summary, _cond_coefs, _cond_imp = controlled_predictor_analysis(
    data, '_intact_01', 'BERM CONDITION (Intact vs Degraded)')

print("\n── Predictor summary ──")
display(_cond_summary.style.format({
    'Univariate p': '{:.4f}',
    'LRT':          '{:.2f}',
    'LRT p':        '{:.4f}',
    '\u0394AIC':         '{:+.1f}',
}).hide(axis='index'))

print("\n── Full model coefficients (sorted by |z|) ──")
display(_cond_coefs.style.format({
    'coef': '{:+.4f}', 'OR': '{:.3f}', 'SE': '{:.4f}',
    'z': '{:.2f}', 'p': '{:.4f}',
}))

print("\n── RF permutation importance (top 10) ──")
display(_cond_imp.head(10).style.format({
    'importance': '{:.4f}', 'std': '{:.4f}',
}))


### Condition results — interpretation

**Full model:** Pseudo R² = 0.080, AIC = 994.4, RF 5-fold CV AUC = 0.739 ± 0.041.

**Predictors that survive controlling:**

| Predictor | Uni p | LRT p | ΔAIC | Interpretation |
|---|---|---|---|---|
| **Berm length** | < 0.001 *** | < 0.001 *** | +30.6 | By far the strongest predictor of condition. Longer berms are more likely to be degraded (OR ≈ 0.992 per metre, i.e. each additional metre of length reduces the odds of intact by ~0.8 %). This is consistent with longer structures having more potential failure points. |
| **Flow accumulation** | 0.011 * | 0.003 ** | +6.7 | Higher upstream contributing area independently increases degradation risk. Effect strengthens after controlling, suggesting it is partially masked by correlated variables in univariate tests. |
| **Soil texture** | < 0.001 *** | 0.017 * | +3.0 | Significant as a group (7 df joint test), though attenuated from the univariate result, indicating partial confounding with landform or clay content. |
| **B-horizon presence** | 0.063 ns | 0.038 * | +2.3 | A classic **suppression** effect — not significant univariately, but significant after controlling. Berms on soils with a B horizon have lower odds of being intact (OR ≈ 0.41). The raw association is masked because B-horizon presence correlates with other predictors (e.g., landform, texture). |

**Predictors that do NOT survive controlling:**

| Predictor | Uni p | LRT p | ΔAIC | Interpretation |
|---|---|---|---|---|
| **Slope** | 0.825 ns | 0.920 ns | −2.0 | Not associated with condition in either framework. Slope does not independently predict structural integrity. |
| **Landform** | 0.204 ns | 0.176 ns | −0.5 | Not significant. Whatever variation landform captures is already explained by the other predictors (slope, texture, flow accumulation). |

**Key takeaway:** Structural condition is driven primarily by berm geometry (length), hydrology (flow accumulation), and soil properties (texture, B-horizon presence). Slope and landform are not independent predictors of condition once the other variables are accounted for.

## Vegetation response (Effective vs Ineffective)


In [ ]:
_eff_summary, _eff_coefs, _eff_imp = controlled_predictor_analysis(
    data, '_effective_01', 'VEGETATION RESPONSE (Effective vs Ineffective)')

print("\n── Predictor summary ──")
display(_eff_summary.style.format({
    'Univariate p': '{:.4f}',
    'LRT':          '{:.2f}',
    'LRT p':        '{:.4f}',
    '\u0394AIC':         '{:+.1f}',
}).hide(axis='index'))

print("\n── Full model coefficients (sorted by |z|) ──")
display(_eff_coefs.style.format({
    'coef': '{:+.4f}', 'OR': '{:.3f}', 'SE': '{:.4f}',
    'z': '{:.2f}', 'p': '{:.4f}',
}))

print("\n── RF permutation importance (top 10) ──")
display(_eff_imp.head(10).style.format({
    'importance': '{:.4f}', 'std': '{:.4f}',
}))


### Vegetation response results — interpretation

**Full model:** Pseudo R² = 0.054, AIC = 1044.4, RF 5-fold CV AUC = 0.667 ± 0.025.

**Predictors that survive controlling:**

| Predictor | Uni p | LRT p | ΔAIC | Interpretation |
|---|---|---|---|---|
| **Slope** | < 0.001 *** | < 0.001 *** | +18.9 | The dominant predictor of vegetation response. Steeper slopes strongly increase the probability of a vegetation response (OR ≈ 1.34 per percentage point). This association holds after controlling and is consistent with greater runoff concentration on steeper terrain. |
| **Soil texture** | < 0.001 *** | 0.011 * | +4.3 | Significant as a group (7 df joint test). Texture classes differ in infiltration capacity and water-holding potential, directly influencing whether diverted water supports vegetation growth. |

**Predictors that do NOT survive controlling:**

| Predictor | Uni p | LRT p | ΔAIC | Interpretation |
|---|---|---|---|---|
| **Landform** | < 0.001 *** | 0.422 ns | −2.3 | Strongly significant univariately but **completely confounded** — the apparent landform effect is absorbed by slope and texture differences between landforms. This is the clearest example of confounding in the dataset. |
| **B-horizon presence** | < 0.001 *** | 0.073 ns | +1.2 | Significant univariately but marginal (p = 0.07) after controlling. The univariate signal likely reflects the correlation between soil development and other soil/terrain properties. The trend (OR ≈ 2.19, favouring vegetation response on B-horizon soils) is suggestive but not conclusive. |
| **Flow accumulation** | 0.019 * | 0.107 ns | +0.6 | Loses significance after controlling. The univariate association is likely mediated by correlated predictors (landform, slope). |
| **Berm length** | 0.609 ns | 0.652 ns | −1.8 | Not associated with vegetation response in either framework. Berm length influences structural condition but not whether the berm produces a vegetation response. |

**Key takeaway:** Vegetation response is driven by terrain slope and soil texture. Landform — despite being the most intuitive grouping variable — adds no independent information once slope and texture are controlled. This suggests that landform acts as a *proxy* for the underlying edaphic and topographic gradients, rather than exerting a direct causal effect on vegetation outcomes.

## Side-by-side comparison


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

# ── Outcome colours (match model figures) ─────────────────────────────────
from constants import MODEL_CLR_CONDITION, MODEL_CLR_VEGRESPONSE, FS

_CLR_C = MODEL_CLR_CONDITION   # blue
_CLR_V = MODEL_CLR_VEGRESPONSE # sage
_FS    = FS                    # uniform font size for this figure

# ── Build combined DataFrame ──────────────────────────────────────────────
_pred_order = ['Berm length', 'Slope', 'Landform', 'Soil texture',
               'B-horizon presence', 'Flow accumulation', 'Clay content']
_viz = pd.DataFrame({'Predictor': _pred_order})
_viz = _viz.merge(
    _cond_summary[['Predictor', 'Univariate p', 'LRT p', '\u0394AIC']].rename(
        columns={'Univariate p': 'Uni_p_cond', 'LRT p': 'LRT_p_cond', '\u0394AIC': 'dAIC_cond'}),
    on='Predictor')
_viz = _viz.merge(
    _eff_summary[['Predictor', 'Univariate p', 'LRT p', '\u0394AIC']].rename(
        columns={'Univariate p': 'Uni_p_eff', 'LRT p': 'LRT_p_eff', '\u0394AIC': 'dAIC_eff'}),
    on='Predictor')

# Sort by max |ΔAIC| ascending so biggest bars are at top
_viz['_sort'] = _viz[['dAIC_cond', 'dAIC_eff']].abs().max(axis=1)
_viz = _viz.sort_values('_sort', ascending=True).reset_index(drop=True)
_n  = len(_viz)
_y  = np.arange(_n)
_bw = 0.35

# ─────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5.5))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1.2], wspace=0.45)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# ── Panel A: ΔAIC grouped horizontal bars ─────────────────────────────────
ax1.barh(_y - _bw/2, _viz['dAIC_cond'], _bw, label='Condition',
         color=_CLR_C, edgecolor='black', linewidth=0.5)
ax1.barh(_y + _bw/2, _viz['dAIC_eff'], _bw, label='Vegetation response',
         color=_CLR_V, edgecolor='black', linewidth=0.5)
ax1.axvline(0, color='black', linewidth=0.8)
ax1.axvline(2, color='grey', linewidth=0.8, linestyle='--')
ax1.text(2.2, _n - 0.5, '\u0394AIC = 2', fontsize=_FS, color='grey', va='bottom')
ax1.set_yticks(_y)
ax1.set_yticklabels(_viz['Predictor'], fontsize=_FS)
ax1.set_xlabel('\u0394AIC  (positive \u2192 predictor improves model)', fontsize=_FS)
ax1.set_title('A.  \u0394AIC (drop-one LRT)', fontsize=_FS, fontweight='bold', loc='left', pad=10)
ax1.legend(fontsize=_FS, loc='lower right')
ax1.tick_params(labelsize=_FS)
ax1.spines[['top', 'right']].set_visible(False)

# ── Panel B: univariate → controlled significance shift ───────────────────
_cap = 5.5  # cap -log10(p) for display
def _nl(p):
    return min(-np.log10(max(p, 1e-10)), _cap)

_thresh = -np.log10(0.05)  # ≈ 1.30

_min_gap = 0.15  # suppress arrow when markers are closer than this

for i, row in _viz.iterrows():
    # Condition (offset down)
    _uc = _nl(row['Uni_p_cond']); _lc = _nl(row['LRT_p_cond'])
    if abs(_lc - _uc) >= _min_gap:
        ax2.annotate('', xy=(_lc, i - 0.15), xytext=(_uc, i - 0.15),
                     arrowprops=dict(arrowstyle='->,head_length=0.6,head_width=0.35', color=_CLR_C, lw=1.8))
    ax2.scatter(_uc, i - 0.15, marker='o', s=55, color=_CLR_C,
                edgecolors='black', linewidth=0.5, zorder=3)
    ax2.scatter(_lc, i - 0.15, marker='s', s=55, color=_CLR_C,
                edgecolors='black', linewidth=0.5, zorder=3)
    # Vegetation response (offset up)
    _ue = _nl(row['Uni_p_eff']); _le = _nl(row['LRT_p_eff'])
    if abs(_le - _ue) >= _min_gap:
        ax2.annotate('', xy=(_le, i + 0.15), xytext=(_ue, i + 0.15),
                     arrowprops=dict(arrowstyle='->,head_length=0.6,head_width=0.35', color=_CLR_V, lw=1.8))
    ax2.scatter(_ue, i + 0.15, marker='o', s=55, color=_CLR_V,
                edgecolors='black', linewidth=0.5, zorder=3)
    ax2.scatter(_le, i + 0.15, marker='s', s=55, color=_CLR_V,
                edgecolors='black', linewidth=0.5, zorder=3)

ax2.axvline(_thresh, color='red', linewidth=0.8, linestyle='--')
ax2.text(_thresh + 0.08, _n - 0.7, 'p = 0.05', fontsize=_FS, color='red', va='bottom')
ax2.set_yticks(_y)
ax2.set_yticklabels(_viz['Predictor'], fontsize=_FS)
ax2.set_xlabel('$-\\log_{10}(p)$   (further right \u2192 more significant)', fontsize=_FS)
ax2.set_title('B.  Univariate \u2192 controlled significance', fontsize=_FS, fontweight='bold', loc='left', pad=10)
ax2.tick_params(labelsize=_FS)
ax2.spines[['top', 'right']].set_visible(False)

_handles = [
    Line2D([0], [0], marker='o', color='grey', markerfacecolor='grey',
           markersize=7, linestyle='None', label='Univariate p'),
    Line2D([0], [0], marker='s', color='grey', markerfacecolor='grey',
           markersize=7, linestyle='None', label='Controlled (LRT) p'),
]
ax2.legend(handles=_handles, fontsize=_FS, loc='lower right')

plt.tight_layout()
plt.show()

**Figure — Independent predictor contributions to berm condition and vegetation response.**
**(A)** ΔAIC from drop-one likelihood-ratio tests. Positive values indicate that removing the focal predictor worsens model fit; the dotted line marks the conventional ΔAIC = 2 threshold. Berm length (ΔAIC = +30.6) and flow accumulation (+6.7) are the strongest independent predictors of structural condition, while slope (+18.9) dominates vegetation response. Soil texture contributes modestly to both outcomes. Landform falls below the threshold for both, indicating it adds no independent information once other variables are controlled.
**(B)** Shift in significance from univariate tests (circles) to controlled likelihood-ratio tests (squares). Arrows connect the two; the red dashed line marks p = 0.05. Key patterns: landform's vegetation-response signal collapses after controlling (strong leftward shift past the significance line), revealing complete confounding with slope and texture. B-horizon presence shows the opposite — a suppression effect for condition, gaining significance only after controlling. Berm length and slope remain highly significant for their respective outcomes in both frameworks, confirming their roles as direct, non-confounded predictors.

In [ ]:
def _sig(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

_comp = _cond_summary[['Predictor']].copy()
_comp['Condition: Uni p']   = _cond_summary['Univariate p'].apply(lambda p: f"{p:.4f} ({_sig(p)})")
_comp['Condition: LRT p']   = _cond_summary['LRT p'].apply(lambda p: f"{p:.4f} ({_sig(p)})")
_comp['Condition: \u0394AIC']   = _cond_summary['\u0394AIC'].apply(lambda v: f"{v:+.1f}")
_comp['Effective: Uni p']   = _eff_summary['Univariate p'].apply(lambda p: f"{p:.4f} ({_sig(p)})")
_comp['Effective: LRT p']   = _eff_summary['LRT p'].apply(lambda p: f"{p:.4f} ({_sig(p)})")
_comp['Effective: \u0394AIC']   = _eff_summary['\u0394AIC'].apply(lambda v: f"{v:+.1f}")

print("Side-by-side: univariate vs controlled significance for each predictor")
print("Positive \u0394AIC = dropping the focal predictor worsens the model\n")
display(_comp.style.hide(axis='index'))


In [ ]:
from pathlib import Path

_out = Path('../data/summary')
_out.mkdir(parents=True, exist_ok=True)

_cond_summary.to_csv(_out / 'controlled_predictors_condition.csv', index=False)
_eff_summary.to_csv(_out / 'controlled_predictors_vegetation.csv', index=False)
_comp.to_csv(_out / 'controlled_predictors_comparison.csv', index=False)

print(f"Saved 3 CSV files to {_out}/")


## Cross-outcome synthesis

### Different predictors dominate each outcome

The two outcomes — structural condition and vegetation response — are governed by **largely non-overlapping** sets of independent predictors:

|  | Condition (intact) | Vegetation response |
|---|---|---|
| **Strong independent predictors** | Berm length (ΔAIC +30.6), flow accumulation (+6.7) | Slope (ΔAIC +18.9) |
| **Moderate independent predictors** | Soil texture (+3.0), B-horizon (+2.3) | Soil texture (+4.3) |
| **Confounded / non-significant** | Slope, landform | Landform, B-horizon, flow accumulation, berm length |

Soil texture is the only predictor that contributes independently to *both* outcomes, though its effect is modest in both cases.

### Confounding patterns

Three important confounding patterns emerge:

1. **Landform is a proxy, not a cause.** Despite strong univariate associations with vegetation response (p < 0.001), landform adds nothing to the controlled model (LRT p = 0.42). Its apparent effect is entirely explained by the slope and texture gradients that distinguish fan terraces, stream terraces, and flood plains.

2. **B-horizon presence is suppressed for condition.** The univariate test (p = 0.063) understates its true effect. After controlling for confounders the B-horizon effect on condition strengthens to p = 0.038. Correlated variables (e.g., landform, texture) mask the independent soil-development signal.

3. **Slope matters for vegetation but not condition; length matters for condition but not vegetation.** These two outcomes respond to fundamentally different site characteristics. Structural integrity is a function of design (length) and hydraulic loading (flow accumulation). Vegetation response is a function of the terrain's ability to concentrate and deliver water (slope) and the soil's capacity to support plant growth (texture).

### Practical implications

- **For predicting structural failure risk:** prioritise berm length, upstream contributing area, and soil texture information.
- **For predicting vegetation benefit:** prioritise slope and soil texture. Landform is a useful field shorthand but adds no information beyond what slope and texture already provide.
- **B-horizon presence** provides a modest but real signal for structural condition that is not captured by the other variables. This may reflect the influence of subsurface soil structure on berm foundation stability.

### Model limitations

- Both models have low pseudo R² (0.05–0.08), indicating that the predictors examined here explain only a small fraction of outcome variance. Unmeasured factors (berm construction quality, local hydrology, maintenance history, age) likely play a large role.
- RF CV AUC values (0.67–0.74) suggest moderate but imperfect discriminative ability.
- All predictors enter the model linearly (or as dummy variables); non-linear or interaction effects are not captured by the logistic framework, though the random forest provides a partial check.

Controlled OR (95 % CI) is the odds ratio for a focal predictor extracted from the full logistic regression (the one with all six predictors in it simultaneously). Because every other predictor is held constant in that model, this OR reflects the independent effect — how much a one-unit increase in the focal variable changes the odds of the outcome, net of confounders.

The Wald p-value is the default significance test that logistic regression software reports next to each coefficient. 
In a logistic regression, each predictor gets a coefficient (β) and a standard error (SE) for that coefficient. The Wald statistic is  the ratio: **z = β / SE**. This follows the same intuition as a t-test — quantifying how many standard errors the coefficient is from zero. If β is large relative to its uncertainty, the ratio is large, and the predictor is likely contributing something real. We then compare z² to a chi-square distribution with 1 degree of freedom (or equivalently, compare z to a standard normal) to get the p-value.
